# Coastal Dynamics — Tarragona Coast, Spain
## A Cloud-Native Time Series Analysis of Shoreline Change

**Open Science in Practice FERS School, Bertinoro, April 2026**

---

### Scientific Question

> Has the coastline near Tarragona (NE Spain) changed in extent and morphology between 2017 and 2024,
> and if so, can those changes be linked to sea level anomaly or sea surface temperature?

### Study Site

The study area (bbox `[0.55, 40.55, 1.05, 40.90]`) covers the Tarragona coastal zone
in Catalonia, NE Spain — including the Costa Daurada beaches, the Ebro Delta margin,
and nearshore Mediterranean waters. The area is sensitive to storm surges, longshore
sediment transport, and seasonal wave climatology.

### Data sources (all open, cloud-native)
| Variable | Source | Access |
|---|---|---|
| Multispectral imagery | Sentinel-2 L2A | Planetary Computer STAC |
| Sea Level Anomaly (SLA) | CMEMS SEALEVEL_GLO | `copernicusmarine` Python client |
| Sea Surface Temperature | CMEMS SST GLO | `copernicusmarine` Python client |

### FAIR compliance
- All data accessed via public APIs — no local files
- Notebook fully reproducible on JupyterHub
- Outputs documented with standard metadata
- Results to be published via Zenodo + STAC catalog

### Section 0 — Environment Setup

In [ ]:
# Core scientific stack
import numpy as np
import xarray as xr
import rioxarray
import pandas as pd

# STAC access
import pystac_client
import planetary_computer

# CMEMS access
import copernicusmarine

# Visualization
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
import panel as pn

hv.extension('bokeh')
pn.extension()

print('All imports successful')

### Section 1 — Study Area Definition

In [ ]:
# Study area — Tarragona coast, NE Spain
BBOX   = [0.55, 40.55, 1.05, 40.90]   # [lon_min, lat_min, lon_max, lat_max]
YEARS  = list(range(2017, 2026))       # 2017–2025 (full Sentinel-2 archive)
CLOUD_COVER_MAX = 20                   # % — Mediterranean, full-year search

# Full-year windows — no dry-season restriction for Spain
DATE_WINDOWS = [(f'{y}-01-01', f'{y}-12-31') for y in YEARS]

print(f'Study area: {BBOX}')
print(f'Time windows: {DATE_WINDOWS[0]} → {DATE_WINDOWS[-1]}')

In [ ]:
import folium

center_lat = (BBOX[1] + BBOX[3]) / 2
center_lon = (BBOX[0] + BBOX[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

folium.Rectangle(
    bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]],
    color='red',
    fill=True,
    fill_opacity=0.2
).add_to(m)

m

### Section 2 — Sentinel-2 Images via STAC

Searching one best (least cloudy) image per year over the full calendar year.

NDWI (McFeeters 1996): `NDWI = (Green − NIR) / (Green + NIR)`
- `NDWI > 0` → Water
- `NDWI < 0` → Land / beach / vegetation

In [ ]:
# Connect to Planetary Computer STAC
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace
)

print('Connected to Planetary Computer STAC catalog')

In [ ]:
# ── Discover which MGRS tiles cover our bbox ──────────────────────────────────
discover = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime='2023-01-01/2023-12-31',
    query={'eo:cloud_cover': {'lt': CLOUD_COVER_MAX}}
)
TILES = sorted({item.properties['s2:mgrs_tile'] for item in discover.items()})
print(f'Tiles covering the study area: {TILES}')

In [ ]:
# ── Search best scene per tile per year ───────────────────────────────────────
best_items = {}   # {year: {tile: item}}

for year, (date_start, date_end) in zip(YEARS, DATE_WINDOWS):
    best_items[year] = {}
    for tile in TILES:
        search = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=BBOX,
            datetime=f'{date_start}/{date_end}',
            query={
                'eo:cloud_cover': {'lt': CLOUD_COVER_MAX},
                's2:mgrs_tile':   {'eq': tile}
            }
        )
        items = list(search.items())
        if items:
            best = min(items, key=lambda x: x.properties['eo:cloud_cover'])
            best_items[year][tile] = best
            print(f"{year}  tile {tile}: {best.datetime.date()} — cloud {best.properties['eo:cloud_cover']:.4f}%")
        else:
            print(f"{year}  tile {tile}: ⚠️  no scenes found")

In [ ]:
# Summary: which tile/date was selected per year
for year in YEARS:
    for tile, item in best_items[year].items():
        print(f"{year}  tile {tile}  |  {item.datetime.date()}")

In [ ]:
# Diagnostic — check CRS and bounds for one tile
year, tile = 2019, TILES[0]
item = best_items[year][tile]
signed = planetary_computer.sign(item)
test = rioxarray.open_rasterio(signed.assets['B04'].href, overview_level=2).squeeze()

print('Image CRS:    ', test.rio.crs)
print('Image bounds: ', test.rio.bounds())
print('Image shape:  ', test.shape)
print('BBOX (WGS84): ', BBOX)
print('Tiles:        ', TILES)

In [ ]:
from rioxarray.merge import merge_arrays
import matplotlib.pyplot as plt

ANALYSIS_YEARS = [2017, 2021, 2024]

def load_band_mosaic(year, band, overview_level=2):
    """Load a band from all tiles for a year, merge into mosaic, clip to BBOX."""
    tiles_data = []
    for tile, item in best_items[year].items():
        signed = planetary_computer.sign(item)
        da = rioxarray.open_rasterio(
            signed.assets[band].href, overview_level=overview_level
        ).squeeze()
        tiles_data.append(da)
    mosaic = merge_arrays(tiles_data) if len(tiles_data) > 1 else tiles_data[0]
    return mosaic.rio.clip_box(*BBOX, crs='EPSG:4326')


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, year in zip(axes, ANALYSIS_YEARS):
    red   = load_band_mosaic(year, 'B04')
    green = load_band_mosaic(year, 'B03')
    blue  = load_band_mosaic(year, 'B02')

    rgb = np.stack([red.values, green.values, blue.values], axis=-1).astype(float)
    p2, p98 = np.percentile(rgb[rgb > 0], [2, 98])
    rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

    date = next(iter(best_items[year].values())).datetime.date()
    ax.imshow(rgb)
    ax.set_title(f"{year}  |  {date}", fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sentinel-2 RGB — Tarragona Coast — 2017 / 2021 / 2024 (mosaicked tiles)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
ndwi_annual = {}
rgb_annual  = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, year in zip(axes, ANALYSIS_YEARS):
    red   = load_band_mosaic(year, 'B04')
    green = load_band_mosaic(year, 'B03')
    blue  = load_band_mosaic(year, 'B02')
    nir   = load_band_mosaic(year, 'B08')

    # RGB visualization
    rgb = np.stack([red.values, green.values, blue.values], axis=-1).astype(float)
    p2, p98 = np.percentile(rgb[rgb > 0], [2, 98])
    rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

    date = next(iter(best_items[year].values())).datetime.date()
    ax.imshow(rgb)
    ax.set_title(f"{year}  |  {date}", fontsize=10, fontweight='bold')
    ax.axis('off')

    # NDWI
    green_f = green.astype(float)
    nir_f   = nir.astype(float)
    ndwi = (green_f - nir_f) / (green_f + nir_f)
    ndwi.name = 'NDWI'
    ndwi.attrs['long_name']   = f'NDWI {year}'
    ndwi.attrs['source_date'] = str(date)

    ndwi_annual[year] = ndwi
    rgb_annual[year]  = xr.concat([red, green, nir], dim='band')

    print(f'✅ {year} — NDWI shape: {ndwi.shape}')

plt.suptitle('Sentinel-2 RGB — Tarragona Coast — 2017 / 2021 / 2024', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('\nSelected years processed.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, year in zip(axes, ANALYSIS_YEARS):
    ndwi = ndwi_annual[year]
    im = ax.imshow(
        ndwi.values,
        cmap='RdYlBu',
        vmin=-1, vmax=1
    )
    ax.set_title(f"NDWI {year}  |  {ndwi.attrs['source_date']}", fontsize=10, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='NDWI')

plt.suptitle('NDWI — Tarragona Coast — 2017 / 2021 / 2024\n(blue = water, red = land/beach)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plots = [
    ndwi_annual[year].hvplot.image(
        x='x', y='y',
        cmap='RdYlBu',
        clim=(-1, 1),
        title=f'NDWI {year}',
        width=700, height=700,
        colorbar=True,
        geo=True,
        tiles='OSM',
    )
    for year in ANALYSIS_YEARS
]

pn.Row(*plots)

In [ ]:
import geopandas as gpd
from rasterio.features import shapes
from shapely.geometry import shape
import warnings

PIXEL_AREA_M2 = 100  # 10m × 10m (overview_level=2 → ~20m, but stored at native res)

land_area   = {}
land_gdf    = {}

for year in ANALYSIS_YEARS:
    ndwi = ndwi_annual[year]

    # Binary land mask: NDWI < 0 → land
    land_mask = (ndwi < 0).astype('uint8')

    # Vectorise land pixels → polygons (rasterio needs a 2-D numpy array + transform)
    transform = ndwi.rio.transform()
    crs       = ndwi.rio.crs

    polys = []
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for geom, val in shapes(land_mask.values, transform=transform):
            if val == 1:
                polys.append(shape(geom))

    gdf = gpd.GeoDataFrame(geometry=polys, crs=crs).dissolve()
    land_gdf[year] = gdf

    # Area in hectares
    gdf_m = gdf.to_crs(gdf.estimate_utm_crs())
    area_ha = gdf_m.area.sum() / 10_000
    n_pixels = int(land_mask.sum())
    land_area[year] = area_ha
    print(f'{year}: {n_pixels} land pixels → {area_ha:.1f} ha')

df_area = (
    pd.DataFrame.from_dict(land_area, orient='index', columns=['land_area_ha'])
    .rename_axis('year')
    .reset_index()
)
df_area

In [ ]:
plots = []
for year in ANALYSIS_YEARS:
    ndwi = ndwi_annual[year]
    gdf  = land_gdf[year].to_crs('EPSG:4326')

    raster = ndwi.hvplot.image(
        x='x', y='y',
        cmap='RdYlBu', clim=(-1, 1),
        alpha=0.7,
        colorbar=True,
        geo=True, tiles='OSM',
        title=f'NDWI + Land polygons — {year}',
        width=550, height=550,
    )

    polys = gdf.hvplot.polygons(
        geo=True,
        fill_color=None,
        line_color='black',
        line_width=1,
    )

    plots.append(raster * polys)

pn.Row(*plots)

### Section 3 — Sea Level Anomaly (CMEMS)

In [ ]:
ds_sla = copernicusmarine.open_dataset(
    dataset_id='cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D',
    variables=['sla'],
    minimum_longitude=BBOX[0],
    maximum_longitude=BBOX[2],
    minimum_latitude=BBOX[1],
    maximum_latitude=BBOX[3],
    start_datetime='2017-01-01',
    end_datetime='2024-12-31',
)

print(ds_sla)

# Annual mean SLA spatially averaged over the Tarragona bbox
sla_annual_mean = (
    ds_sla['sla']
    .mean(dim=['latitude', 'longitude'])
    .resample(time='YE')
    .mean()
)

df_sla = sla_annual_mean.to_dataframe().reset_index()
df_sla['year'] = df_sla['time'].dt.year
df_sla = df_sla[['year', 'sla']]
df_sla

### Section 4 — Significant Wave Height (CMEMS)

Wave energy is the primary driver of shoreline change along the Tarragona coast (beach erosion, longshore sediment transport, Ebro Delta margin dynamics). Annual mean significant wave height (Hs) from the CMEMS global wave reanalysis is used here as the forcing variable.

In [ ]:
ds_waves = copernicusmarine.open_dataset(
    dataset_id='cmems_mod_glo_wav_my_0.2deg_PT3H-i',
    variables=['VHM0'],           # significant wave height (m)
    minimum_longitude=BBOX[0],
    maximum_longitude=BBOX[2],
    minimum_latitude=BBOX[1],
    maximum_latitude=BBOX[3],
    start_datetime='2017-01-01',
    end_datetime='2024-12-31',
)

print(ds_waves)

# Storm season: Oct–Feb (peak Mediterranean wave energy)
hs = ds_waves['VHM0']
hs_storm = hs.sel(time=hs.time.dt.month.isin([10, 11, 12, 1, 2]))

hs_annual = (
    hs_storm
    .mean(dim=['latitude', 'longitude'])
    .resample(time='YE')
    .mean()
)

df_hs = hs_annual.to_dataframe().reset_index()
df_hs['year'] = df_hs['time'].dt.year
df_hs = df_hs[['year', 'VHM0']].rename(columns={'VHM0': 'hs_m'})
df_hs

In [ ]:
df = df_area.merge(df_sla, on='year').merge(df_hs, on='year')
df['year'] = df['year'].astype(int)

def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

df['land_area_norm'] = normalize(df['land_area_ha'])
df['sla_norm']       = normalize(df['sla'])
df['hs_norm']        = normalize(df['hs_m'])

print('Integrated dataset:')
df

### Section 5 — Interactive Dashboard

Two-panel dashboard linked by a year selector:
- **Left:** NDWI map with land polygon overlay for the selected year
- **Right:** Time series of land area, SLA, and significant wave height with a year marker

In [ ]:
year_slider = pn.widgets.DiscreteSlider(
    name='Year',
    options=ANALYSIS_YEARS,
    value=ANALYSIS_YEARS[0]
)

# ── MAP PANEL: NDWI + land polygon overlay ───────────────────────────────────
@pn.depends(year_slider.param.value)
def ndwi_map(year):
    ndwi = ndwi_annual[year]
    gdf  = land_gdf[year].to_crs('EPSG:4326')
    date = ndwi.attrs['source_date']

    base = ndwi.hvplot.image(
        x='x', y='y',
        cmap='RdYlBu', clim=(-1, 1),
        title=f'NDWI — Tarragona {year}  |  {date}',
        width=520, height=460,
        colorbar=True,
        clabel='NDWI (blue=water, red=land)',
        geo=True, tiles='OSM',
    )

    polys = gdf.hvplot.polygons(
        geo=True,
        fill_color=None,
        line_color='black',
        line_width=1,
    )

    return base * polys

# ── TIME SERIES PANEL ────────────────────────────────────────────────────────
@pn.depends(year_slider.param.value)
def timeseries_panel(year):
    area_line = (
        df.hvplot.line(
            x='year', y='land_area_ha',
            color='#F5A623', line_width=2.5,
            label='Land area (ha)',
            width=400, height=180,
            ylabel='Area (ha)',
        )
        * df.hvplot.scatter(x='year', y='land_area_ha', color='#F5A623', size=80)
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    sla_line = (
        df_sla.hvplot.line(
            x='year', y='sla',
            color='#2E86C1', line_width=2.5,
            label='Sea level anomaly (m)',
            width=400, height=180,
            ylabel='SLA (m)',
        )
        * df_sla.hvplot.scatter(x='year', y='sla', color='#2E86C1', size=40)
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    hs_line = (
        df_hs.hvplot.line(
            x='year', y='hs_m',
            color='#1ABC9C', line_width=2.5,
            label='Significant wave height (m)',
            width=400, height=180,
            ylabel='Hs (m)',
        )
        * df_hs.hvplot.scatter(x='year', y='hs_m', color='#1ABC9C', size=40)
        * hv.VLine(year).opts(color='gray', line_dash='dashed')
    )

    return pn.Column(
        pn.pane.Markdown('### Land area · Sea level anomaly · Wave height'),
        area_line,
        sla_line,
        hs_line,
    )

# ── ASSEMBLE DASHBOARD ───────────────────────────────────────────────────────
header = pn.pane.Markdown(
    '# Tarragona Coast — Shoreline Dynamics 2017 / 2021 / 2024\n'
    '**Data:** Sentinel-2 L2A (Planetary Computer) · CMEMS SLA · CMEMS Wave reanalysis  '
    '| **Index:** NDWI (McFeeters 1996) | **Resolution:** ~20 m',
    sizing_mode='stretch_width'
)

dashboard = pn.Column(
    header,
    pn.Row(pn.Column('### Select year', year_slider)),
    pn.Row(
        pn.panel(ndwi_map),
        pn.panel(timeseries_panel),
    ),
)

dashboard.servable()
dashboard